# 📊 QLoRA Fine-Tuning Project — Notebook 4: Ablation Study & Analysis
**Goal:** Compare all methods and understand where fine-tuning helps vs prompt engineering

## Methods Compared
| Method | Type | Training Required |
|--------|------|-------------------|
| Zero-Shot | Prompt Engineering | ❌ None |
| Few-Shot | Prompt Engineering | ❌ None |
| Optimized Prompt | Prompt Engineering | ❌ None |
| QLoRA Fine-tuned | Fine-Tuning | ✅ 3 Epochs |

## Analysis Includes
1. ROUGE score comparison across all 4 methods
2. Statistical significance testing (paired t-test)
3. Length analysis (predicted vs reference summary lengths)
4. Qualitative examples — where fine-tuning wins/loses
5. Error analysis
6. Final conclusions

**Runtime:** ~15-20 minutes (no GPU needed)  
**Input:** `qlora-baseline-results` + `qlora-finetuned-results` datasets

In [ ]:
# ══════════════════════════════════════════════════════
# 📦 CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════
!pip install -q evaluate rouge-score matplotlib seaborn pandas numpy scipy
print("✅ Dependencies installed")

In [ ]:
# ══════════════════════════════════════════════════════
# ⚙️ CELL 2: Paths & Setup
# ══════════════════════════════════════════════════════
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Input paths ────────────────────────────────────────
BASELINE_DIR  = "/kaggle/input/datasets/pranitsaundankar/qlora-baseline-results/qlora_project/results"
FINETUNED_DIR = "/kaggle/input/datasets/pranitsaundankar/qlora-checkpoint-backup/qlora_project/results"

# ── Output path ────────────────────────────────────────
BASE_PATH   = "/kaggle/working/qlora_project"
RESULTS_DIR = f"{BASE_PATH}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Verify all required files exist ───────────────────
required_files = [
    f"{BASELINE_DIR}/baseline_results.csv",
    f"{BASELINE_DIR}/baseline_predictions.csv",
    f"{FINETUNED_DIR}/finetuned_results.csv",
    f"{FINETUNED_DIR}/finetuned_predictions.csv",
]
print("🔍 Checking required files:")
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"   {status} {f.split('/')[-1]}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All files found — ready to run!")
else:
    print("\n❌ Some files missing — check paths above")

# ── Plot style ─────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "Zero-Shot":        "#5B8DB8",
    "Few-Shot":         "#E8844A",
    "Optimized Prompt": "#6BAE75",
    "QLoRA Fine-tuned": "#B05CD3"
}

print(f"\n   Baseline  dir: {BASELINE_DIR}")
print(f"   Finetuned dir: {FINETUNED_DIR}")

## 📥 Load All Results

In [ ]:
# ══════════════════════════════════════════════════════
# 📥 CELL 3: Load All Results & Predictions
# ══════════════════════════════════════════════════════
print("📥 Loading results...\n")

# ── ROUGE scores ───────────────────────────────────────
baseline_results  = pd.read_csv(os.path.join(BASELINE_DIR,  "baseline_results.csv"))
finetuned_results = pd.read_csv(os.path.join(FINETUNED_DIR, "finetuned_results.csv"))
all_results       = pd.concat([baseline_results, finetuned_results], ignore_index=True)

print("📊 ROUGE Scores:")
print(all_results.to_string(index=False))

# ── Predictions ────────────────────────────────────────
baseline_preds  = pd.read_csv(os.path.join(BASELINE_DIR,  "baseline_predictions.csv"))
finetuned_preds = pd.read_csv(os.path.join(FINETUNED_DIR, "finetuned_predictions.csv"))

print(f"\n✅ Baseline predictions  : {len(baseline_preds):,} rows, cols: {list(baseline_preds.columns)}")
print(f"✅ Finetuned predictions  : {len(finetuned_preds):,} rows, cols: {list(finetuned_preds.columns)}")

# ── Merge on reference ─────────────────────────────────
assert len(baseline_preds) == len(finetuned_preds), "Sample count mismatch!"
references = baseline_preds["reference"].tolist()

predictions = {
    "Zero-Shot":        baseline_preds["zero_shot"].tolist(),
    "Few-Shot":         baseline_preds["few_shot"].tolist(),
    "Optimized Prompt": baseline_preds["optimized"].tolist(),
    "QLoRA Fine-tuned": finetuned_preds["finetuned"].tolist(),
}
print(f"\n✅ All predictions loaded ({len(references)} samples each)")

## 📊 ROUGE Score Comparison

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 4: Main ROUGE Comparison Plot
# ══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))

methods = all_results["method"].tolist()
x       = np.arange(len(methods))
width   = 0.25

colors_list = [COLORS.get(m, "#888") for m in methods]

b1 = ax.bar(x - width,   all_results["rouge1"], width, label="ROUGE-1", alpha=0.85, color=[COLORS.get(m, "#888") for m in methods])
b2 = ax.bar(x,           all_results["rouge2"], width, label="ROUGE-2", alpha=0.6,  color=[COLORS.get(m, "#888") for m in methods])
b3 = ax.bar(x + width,   all_results["rougeL"], width, label="ROUGE-L", alpha=0.4,  color=[COLORS.get(m, "#888") for m in methods])

# Value labels
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f"{h:.3f}", ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_xlabel("Method", fontsize=12)
ax.set_ylabel("ROUGE Score", fontsize=12)
ax.set_title("All Methods — ROUGE Score Comparison\nCNN/DailyMail News Summarization", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, all_results["rouge1"].max() + 0.08)
ax.grid(axis='y', alpha=0.4)

# Highlight best method
best_idx = all_results["rouge1"].idxmax()
ax.axvspan(best_idx - 0.4, best_idx + 0.4, alpha=0.08, color='gold', label='Best Method')

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/rouge_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved rouge_comparison.png")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 5: Improvement Over Zero-Shot (Heatmap)
# ══════════════════════════════════════════════════════
zero_shot_row = all_results[all_results["method"] == "Zero-Shot"].iloc[0]

improvements = []
for _, row in all_results.iterrows():
    improvements.append({
        "method": row["method"],
        "ROUGE-1 Δ": row["rouge1"] - zero_shot_row["rouge1"],
        "ROUGE-2 Δ": row["rouge2"] - zero_shot_row["rouge2"],
        "ROUGE-L Δ": row["rougeL"] - zero_shot_row["rougeL"],
    })

imp_df = pd.DataFrame(improvements).set_index("method")

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    imp_df,
    annot=True,
    fmt="+.4f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 11}
)
ax.set_title("Improvement Over Zero-Shot Baseline (Δ ROUGE)", fontsize=13, fontweight='bold')
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/improvement_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved improvement_heatmap.png")

## 📐 Per-Sample Analysis & Statistical Tests

In [ ]:
# ══════════════════════════════════════════════════════
# 📐 CELL 6: Per-Sample ROUGE Scores
# ══════════════════════════════════════════════════════
import evaluate
rouge = evaluate.load("rouge")

def per_sample_rouge1(preds, refs):
    scores = []
    for p, r in zip(preds, refs):
        # ✅ Handle nan/empty predictions
        p = str(p) if p is not None and str(p) != "nan" else "no summary"
        r = str(r) if r is not None and str(r) != "nan" else "no reference"
        s = rouge.compute(predictions=[p], references=[r])
        scores.append(s["rouge1"])
    return scores

# ── Check for nan values in predictions ───────────────
print("🔍 Checking for NaN values:")
for method, preds in predictions.items():
    nan_count = sum(1 for p in preds if str(p) == "nan" or p is None)
    print(f"   {method}: {nan_count} NaN values out of {len(preds)}")

print("\nComputing per-sample ROUGE-1 scores (takes ~5 min)...")
per_sample = {}
for method, preds in predictions.items():
    print(f"   {method}...")
    per_sample[method] = per_sample_rouge1(preds, references)

print("✅ Per-sample scores computed")

# Build dataframe
per_sample_df = pd.DataFrame(per_sample)
print(f"\nPer-sample stats:")
print(per_sample_df.describe().round(4))

In [ ]:
# ══════════════════════════════════════════════════════
# 📐 CELL 7: Statistical Significance Tests
# ══════════════════════════════════════════════════════
print("📐 Paired t-tests vs Zero-Shot (ROUGE-1)\n")
print(f"{'Comparison':<35} {'t-stat':>8} {'p-value':>10} {'Significant':>12}")
print("─" * 70)

zs_scores = per_sample["Zero-Shot"]

for method in ["Few-Shot", "Optimized Prompt", "QLoRA Fine-tuned"]:
    t_stat, p_val = stats.ttest_rel(per_sample[method], zs_scores)
    sig = "✅ YES" if p_val < 0.05 else "❌ NO"
    print(f"{method:<35} {t_stat:>8.3f} {p_val:>10.4f} {sig:>12}")

print("\nNote: p < 0.05 = statistically significant improvement")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 8: Distribution Plot (Violin)
# ══════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 6))

data_for_plot = []
for method, scores in per_sample.items():
    for s in scores:
        data_for_plot.append({"Method": method, "ROUGE-1": s})

plot_df = pd.DataFrame(data_for_plot)

violin_colors = [COLORS.get(m, "#888") for m in per_sample.keys()]
parts = ax.violinplot(
    [per_sample[m] for m in per_sample.keys()],
    positions=range(len(per_sample)),
    showmeans=True,
    showmedians=True
)

for i, (pc, color) in enumerate(zip(parts['bodies'], violin_colors)):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)

ax.set_xticks(range(len(per_sample)))
ax.set_xticklabels(list(per_sample.keys()), fontsize=11)
ax.set_xlabel("Method", fontsize=12)
ax.set_ylabel("Per-Sample ROUGE-1", fontsize=12)
ax.set_title("Per-Sample ROUGE-1 Distribution by Method", fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/rouge_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved rouge_distribution.png")

## 📏 Summary Length Analysis

In [ ]:
# ══════════════════════════════════════════════════════
# 📏 CELL 9: Summary Length Analysis
# ══════════════════════════════════════════════════════
def word_count(texts):
    # ✅ Handle nan/float values
    return [len(str(t).split()) if str(t) != "nan" else 0 for t in texts]

ref_lengths = word_count(references)
length_data = {"Reference": ref_lengths}
for method, preds in predictions.items():
    length_data[method] = word_count(preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean lengths bar chart
means = {k: np.mean(v) for k, v in length_data.items()}
stds  = {k: np.std(v)  for k, v in length_data.items()}

bars = axes[0].bar(
    means.keys(), means.values(),
    color=["#888888"] + [COLORS.get(m, "#888") for m in predictions.keys()],
    alpha=0.85
)
axes[0].errorbar(range(len(means)), list(means.values()),
                 yerr=list(stds.values()), fmt='none', color='black', capsize=4)
axes[0].set_xlabel("Method")
axes[0].set_ylabel("Mean Word Count")
axes[0].set_title("Average Summary Length by Method")
axes[0].tick_params(axis='x', rotation=30)
for bar, (k, v) in zip(bars, means.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{v:.1f}", ha='center', fontsize=9)

# Length distribution boxplot
axes[1].boxplot(
    list(length_data.values()),
    labels=list(length_data.keys()),
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", alpha=0.7)
)
axes[1].set_xlabel("Method")
axes[1].set_ylabel("Word Count")
axes[1].set_title("Summary Length Distribution")
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle("Summary Length Analysis", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/length_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved length_analysis.png")

print("\n📏 Mean word counts:")
for k, v in means.items():
    print(f"   {k:<25}: {v:.1f} ± {stds[k]:.1f}")

## 🔍 Qualitative Analysis

In [ ]:
# ══════════════════════════════════════════════════════
# 🔍 CELL 10: Qualitative Examples
# ══════════════════════════════════════════════════════
print("🔍 Qualitative Comparison — 5 Examples\n")
print("=" * 90)

for i in range(5):
    print(f"\n📌 Example {i+1}")
    print(f"ARTICLE         : {references[i][:200]}...")
    for method, preds in predictions.items():
        label = f"{method:<20}"
        print(f"{label}: {preds[i][:120]}")
    print("─" * 90)

In [ ]:
# ══════════════════════════════════════════════════════
# 🔍 CELL 11: Best & Worst Cases for Fine-tuning
# ══════════════════════════════════════════════════════
# Find cases where fine-tuning helped most vs hurt most
ft_scores = np.array(per_sample["QLoRA Fine-tuned"])
op_scores = np.array(per_sample["Optimized Prompt"])  # best baseline
diff = ft_scores - op_scores

best_indices  = np.argsort(diff)[-5:][::-1]   # top 5 improvements
worst_indices = np.argsort(diff)[:5]           # top 5 regressions

print("🏆 TOP 5 cases where Fine-Tuning HELPS most:\n")
for i, idx in enumerate(best_indices):
    print(f"  Case {i+1} (Δ ROUGE-1 = +{diff[idx]:.3f})")
    print(f"  Reference  : {references[idx][:100]}")
    print(f"  Optimized  : {predictions['Optimized Prompt'][idx][:100]}")
    print(f"  Fine-tuned : {predictions['QLoRA Fine-tuned'][idx][:100]}")
    print()

print("\n❌ TOP 5 cases where Fine-Tuning HURTS most:\n")
for i, idx in enumerate(worst_indices):
    print(f"  Case {i+1} (Δ ROUGE-1 = {diff[idx]:.3f})")
    print(f"  Reference  : {references[idx][:100]}")
    print(f"  Optimized  : {predictions['Optimized Prompt'][idx][:100]}")
    print(f"  Fine-tuned : {predictions['QLoRA Fine-tuned'][idx][:100]}")
    print()

## 📊 Final Summary Dashboard

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 12: Final Ablation Summary Plot
# ══════════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 10))
fig.suptitle("QLoRA Fine-Tuning Ablation Study\nCNN/DailyMail News Summarization — Mistral-7B", 
             fontsize=15, fontweight='bold', y=0.98)

# ── Plot 1: ROUGE comparison ────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
method_colors = [COLORS.get(m, "#888") for m in all_results["method"]]
bars = ax1.bar(all_results["method"], all_results["rouge1"], color=method_colors, alpha=0.85)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{bar.get_height():.3f}", ha='center', fontsize=9, fontweight='bold')
ax1.set_title("ROUGE-1 Scores", fontweight='bold')
ax1.set_ylabel("ROUGE-1")
ax1.tick_params(axis='x', rotation=25, labelsize=9)
ax1.set_ylim(0, all_results["rouge1"].max() + 0.06)

# ── Plot 2: All ROUGE metrics ───────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
metrics = ["rouge1", "rouge2", "rougeL"]
metric_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
x = np.arange(len(all_results["method"]))
w = 0.25
for i, (m, label) in enumerate(zip(metrics, metric_labels)):
    ax2.bar(x + (i-1)*w, all_results[m], w, label=label, alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(all_results["method"], rotation=25, fontsize=9)
ax2.set_title("All ROUGE Metrics", fontweight='bold')
ax2.set_ylabel("Score")
ax2.legend(fontsize=9)

# ── Plot 3: Mean summary lengths ────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
all_methods = ["Reference"] + list(predictions.keys())
length_means = [np.mean(length_data[m]) for m in all_methods]
all_colors = ["#888888"] + [COLORS.get(m, "#888") for m in predictions.keys()]
bars3 = ax3.bar(all_methods, length_means, color=all_colors, alpha=0.85)
for bar in bars3:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{bar.get_height():.0f}w", ha='center', fontsize=9)
ax3.set_title("Mean Summary Length (Words)", fontweight='bold')
ax3.set_ylabel("Word Count")
ax3.tick_params(axis='x', rotation=25, labelsize=9)

# ── Plot 4: Method ranking ──────────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
ranking_data = all_results.set_index("method")[["rouge1", "rouge2", "rougeL"]]
ranking_data.plot(kind="barh", ax=ax4, alpha=0.85)
ax4.set_title("Method Ranking (All ROUGE)", fontweight='bold')
ax4.set_xlabel("Score")
ax4.legend(fontsize=9, loc='lower right')
ax4.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/ablation_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved ablation_summary.png")

In [ ]:
# ══════════════════════════════════════════════════════
# 💾 CELL 13: Save Final Summary JSON
# ══════════════════════════════════════════════════════
# Gather statistical test results
stat_tests = {}
for method in ["Few-Shot", "Optimized Prompt", "QLoRA Fine-tuned"]:
    t_stat, p_val = stats.ttest_rel(per_sample[method], per_sample["Zero-Shot"])
    stat_tests[f"{method} vs Zero-Shot"] = {
        "t_statistic": round(t_stat, 4),
        "p_value": round(p_val, 6),
        "significant": bool(p_val < 0.05)
    }

final_summary = {
    "project": "QLoRA Fine-Tuning vs Prompt Engineering",
    "dataset": "CNN/DailyMail (cnn_dailymail, 3.0.0)",
    "base_model": "mistralai/Mistral-7B-v0.1",
    "test_samples": len(references),
    "rouge_scores": all_results.set_index("method")[["rouge1","rouge2","rougeL"]].to_dict(),
    "statistical_tests": stat_tests,
    "length_analysis": {
        m: {"mean": round(np.mean(v), 2), "std": round(np.std(v), 2)}
        for m, v in length_data.items()
    },
    "best_method_rouge1": all_results.loc[all_results["rouge1"].idxmax(), "method"],
    "best_rouge1": float(all_results["rouge1"].max()),
    "finetuning_improvement_over_zero_shot": {
        "rouge1": float(
            all_results.loc[all_results["method"]=="QLoRA Fine-tuned","rouge1"].values[0] -
            all_results.loc[all_results["method"]=="Zero-Shot","rouge1"].values[0]
        ),
        "rouge2": float(
            all_results.loc[all_results["method"]=="QLoRA Fine-tuned","rouge2"].values[0] -
            all_results.loc[all_results["method"]=="Zero-Shot","rouge2"].values[0]
        ),
    }
}

summary_path = os.path.join(RESULTS_DIR, "ablation_summary.json")
with open(summary_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print("📊 Final Ablation Summary:")
print(json.dumps(final_summary, indent=2))
print(f"\n✅ Saved ablation_summary.json")

## 📝 Conclusions

In [ ]:
# ══════════════════════════════════════════════════════
# 📝 CELL 14: Print Conclusions
# ══════════════════════════════════════════════════════
best_method = all_results.loc[all_results["rouge1"].idxmax(), "method"]
best_r1 = all_results["rouge1"].max()
zs_r1   = all_results.loc[all_results["method"]=="Zero-Shot","rouge1"].values[0]
ft_r1   = all_results.loc[all_results["method"]=="QLoRA Fine-tuned","rouge1"].values[0]
improvement = ((ft_r1 - zs_r1) / zs_r1) * 100

print("=" * 70)
print("📝 CONCLUSIONS")
print("=" * 70)
print(f"\n🏆 Best method overall: {best_method} (ROUGE-1 = {best_r1:.4f})")
print(f"\n📈 QLoRA Fine-tuning vs Zero-Shot:")
print(f"   ROUGE-1 change: {ft_r1 - zs_r1:+.4f} ({improvement:+.1f}%)")
print(f"\n🔬 Statistical Significance:")
for test_name, result in stat_tests.items():
    status = "✅ Significant" if result["significant"] else "❌ Not Significant"
    print(f"   {test_name}: p={result['p_value']:.4f} — {status}")
print(f"\n📊 All output files saved to: {RESULTS_DIR}/")
print("=" * 70)

## ✅ Project Complete!

### Files for GitHub:
| File | Description |
|------|-------------|
| `ablation_summary.png` | Main comparison dashboard |
| `rouge_comparison.png` | ROUGE scores bar chart |
| `improvement_heatmap.png` | Improvement over zero-shot |
| `rouge_distribution.png` | Per-sample distributions |
| `length_analysis.png` | Summary length comparison |
| `training_curves.png` | QLoRA training loss curves |
| `baseline_results.csv` | All baseline ROUGE scores |
| `finetuned_results.csv` | Fine-tuned model ROUGE scores |
| `ablation_summary.json` | Full summary with stats |
| `training_summary.json` | Training configuration |

### Project Summary for LinkedIn/GitHub:
> Fine-tuned **Mistral-7B** using **QLoRA** (4-bit quantization + LoRA adapters) on CNN/DailyMail news summarization. Compared against zero-shot, few-shot, and optimized prompt engineering baselines. Trained for 3 epochs on 8,000 samples using Kaggle T4 GPU with a checkpoint-resume system. Evaluated on 1,000 test samples using ROUGE metrics with statistical significance testing.